####import package and libraries

In [14]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

##Load MNIST and preprocess

In [15]:
(X_train_full, y_train_full), _ = tf.keras.datasets.mnist.load_data()
X = X_train_full.reshape(-1, 28*28) / 255.0
y = to_categorical(y_train_full)

# Use only 3000 samples for speed
X_small, y_small = X[:3000], y[:3000]
X_train, X_val, y_train, y_val = train_test_split(X_small, y_small, test_size=0.2, random_state=42)


In [16]:
def decode_particle(p):
    neurons = [int(p[0]), int(p[1]), int(p[2])]
    activation = ['relu', 'tanh', 'sigmoid'][int(p[3])]
    dropout = float(p[4])
    return neurons, activation, dropout

##Define fitness function for PSO

In [17]:
def fitness(particle):
    neurons, activation, dropout = decode_particle(particle)

    model = Sequential()
    model.add(Dense(neurons[0], activation=activation, input_shape=(784,)))
    model.add(Dropout(dropout))
    model.add(Dense(neurons[1], activation=activation))
    model.add(Dropout(dropout))
    model.add(Dense(neurons[2], activation=activation))
    model.add(Dropout(dropout))
    model.add(Dense(10, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(X_train, y_train, epochs=3, batch_size=128, verbose=0,
                        validation_data=(X_val, y_val))

    return -history.history['val_accuracy'][-1]


##Set PSO hyperparameters

In [18]:
# TODO: PSO hyperparameters
n_particles = 4
n_iterations = 5
w, c1, c2 = 0.5, 1.5, 1.5

bounds = np.array([
    [32, 256],
    [32, 256],
    [16, 128],
    [0, 3],
    [0.0, 0.5]
])

# TODO: PSO setup
positions = np.random.uniform(bounds[:,0], bounds[:,1], (n_particles, 5))
velocities = np.zeros_like(positions)
pbest = positions.copy()
pbest_fitness = np.array([fitness(pos) for pos in positions])
gbest_idx = np.argmin(pbest_fitness)
gbest = pbest[gbest_idx]
gbest_fitness = pbest_fitness[gbest_idx]
history = [gbest_fitness]


/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


##PSO main loop

In [19]:
for t in range(n_iterations):
    for i in range(n_particles):
        r1, r2 = np.random.rand(2)
        velocities[i] = (w * velocities[i]
                         + c1 * r1 * (pbest[i] - positions[i])
                         + c2 * r2 * (gbest - positions[i]))
        positions[i] += velocities[i]
        positions[i] = np.clip(positions[i], bounds[:,0], bounds[:,1])

        score = fitness(positions[i])
        if score < pbest_fitness[i]:
            pbest[i] = positions[i]
            pbest_fitness[i] = score
            if score < gbest_fitness:
                gbest = positions[i]
                gbest_fitness = score
    history.append(gbest_fitness)


##Show best result

In [20]:

neurons, activation, dropout = decode_particle(gbest)
print("Best Architecture:")
print("Neurons:", neurons)
print("Activation:", activation)
print("Dropout:", dropout)
print("Validation Accuracy:", -gbest_fitness)


Best Architecture:
Neurons: [170, 121, 66]
Activation: tanh
Dropout: 0.2005226040478139
Validation Accuracy: 0.9016666412353516
